# Laboration 1

## imports

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.utils.data import random_split
import data_loading_code
from torch.utils.data import TensorDataset, DataLoader
import pandas as pd
from transformers import DistilBertTokenizer, DistilBertModel

import wandb

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Rasmus\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## Load Data

Might be wise to not load the 25K file if it isn't needed then only load the regular one.

In [ ]:
num_epochs = 50

In [2]:
from importlib import reload
reload(data_loading_code)
from data_loading_code import *#, load_data_LSTM

# train_x, train_y, val_x, val_y, vocab_size = load_data("amazon_cells_labelled.txt")
# train_L_x, train_L_y, val_L_x, val_L_y = load_data("amazon_cells_labelled_LARGE_25K.txt")
# print(train_x.shape)

# print(train_L_x)

train_dataset_LSTM = TensorDataset(train_x_tensor, train_y_tensor)
train_loader_LSTM = DataLoader(train_dataset_LSTM, batch_size=32, shuffle=True)

val_dataset_LSTM = TensorDataset(validation_x_tensor, validation_y_tensor)
val_loader_LSTM = DataLoader(val_dataset_LSTM, batch_size=32)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Rasmus\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
input_size = train_x_tensor.shape[1]
print(input_size)

class LSTM(nn.Module):
    def __init__(self, hidden_size = 128):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first= True)
        self.dropout = nn.Dropout(0.15)
        self.fc = nn.Linear(hidden_size, 2)

    def forward(self, x):
        x = x.unsqueeze(1)  # (batch, 1, features)
        out, (hidden, cell) = self.lstm(x)
        return self.fc(hidden[-1])

7277


## Task 1.1.1
A simple neural network composed of linear layers. You may incorporate activation
functions, dropout, and other complementary layers as needed.

In [ ]:
model_1_1 = nn.Sequential(
    nn.Linear(input_size, 128),
    nn.ReLU(),
    nn.Dropout(0.15),
    nn.Linear(128,2) # Negative or positive review
)
# wandb.init(project="Task 1.1.1: Simple ANN", name="ANN-1")
criterion = nn.CrossEntropyLoss()
#optimizer = torch.optim.SGD(model_LSTM.parameters(), lr=0.0001)
optimizer = torch.optim.Adam(model_1_1.parameters(), lr=0.0001)
num_epochs = 10

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor = 0.1,
        patience = 3 # Wait x amount of epochs before reducing lr
    )

best_val_loss = float('inf')


for epoch in range(num_epochs):
    model_1_1.train()

    train_running_loss = 0.0
    for batch_x, batch_y in train_loader_LSTM:
#------------------------- TRAINING -------------------------
        optimizer.zero_grad()
        output = model_1_1(batch_x)
        loss = criterion(output, batch_y)
        loss.backward()
        optimizer.step()
        train_running_loss+=loss.item()
    average_train_loss = train_running_loss/len(train_loader_LSTM)

#------------------------- VALIDATION -------------------------
    model_1_1.eval()
    val_running_loss = 0.0
    with torch.no_grad():
        for batch_x, batch_y in val_loader_LSTM:
            output = model_1_1(batch_x)
            val_loss = criterion(output, batch_y)
            val_running_loss += val_loss.item()
        average_val_loss = val_running_loss/len(val_loader_LSTM)
    scheduler.step(average_val_loss)

#------------------------- VISUALIZATION -------------------------
    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"train loss: {average_train_loss:.3f}, "
        f"val loss: {average_val_loss:.3f}, "
        f"lr: {optimizer.param_groups[0]['lr']:.6f}"
    )
#------------------------- SAVING BEST MODEL -------------------------
    if average_val_loss < best_val_loss:
        best_val_loss = average_val_loss
        torch.save(model_1_1.state_dict(), "BestModel_1_1.pth")
        print("Saved The Best Performing Model")


# wandb.finish()

In [ ]:
#Test the model
model_1_1.load_state_dict(torch.load("BestModel_1_1.pth"))
model_1_1.eval()

test_loss = 0.0
correct = 0
total = 0

with torch.no_grad():
    for batch_x, batch_y in val_loader_LSTM:
        output = model_1_1(batch_x)
        val_loss = criterion(output, batch_y)
        test_loss += val_loss.item()

        HighestValue, predicted = torch.max(output, 1)
        total += batch_y.size(0)
        correct += (predicted == batch_y).sum().item()

average_test_loss = test_loss / len(val_loader_LSTM)
test_acc = 100*(correct/total)


print(f"Test loss: {average_test_loss:.3f}")
print(f"Test accuracy: {test_acc:.2f}%")

# wandb.finish()

## Task 1.1.2
A neural network based on LSTM or bidirectional LSTM (Bi-LSTM) layers.

In [ ]:
# wandb.init(project="Task 1.1.2: LSTM RNN", name="RNN-1")

model_LSTM = LSTM()
criterion = nn.CrossEntropyLoss()
# optimizer = torch.optim.SGD(model_LSTM.parameters(), lr=0.0001)
optimizer = torch.optim.Adam(model_LSTM.parameters(), lr=0.001)
num_epochs = 10

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor = 0.1,
        patience = 3 # Wait x amount of epochs before reducing lr
    )

best_val_loss_LSTM = float('inf')


for epoch in range(num_epochs):
    model_LSTM.train()

    running_train_loss_LSTM = 0.0
    for batch_x, batch_y in train_loader_LSTM:
#------------------------- TRAINING -------------------------
        optimizer.zero_grad()
        output = model_LSTM(batch_x)
        loss_LSTM = criterion(output, batch_y)
        loss_LSTM.backward()
        optimizer.step()
        running_train_loss_LSTM += loss_LSTM.item()
    average_train_loss_LSTM = running_train_loss_LSTM/len(train_loader_LSTM)

#------------------------- VALIDATION -------------------------
    model_LSTM.eval()
    val_running_loss_LSTM = 0.0
    with torch.no_grad():
        for batch_x, batch_y in val_loader_LSTM:
            output = model_LSTM(batch_x)
            val_loss_LSTM = criterion(output, batch_y)
            val_running_loss_LSTM += val_loss_LSTM.item()
        average_val_loss_LSTM = val_running_loss_LSTM/len(val_loader_LSTM)
    scheduler.step(average_val_loss_LSTM)

#------------------------- VISUALIZATION -------------------------
    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"train loss: {average_train_loss_LSTM:.3f}, "
        f"val loss: {average_val_loss_LSTM:.3f}, "
        f"lr: {optimizer.param_groups[0]['lr']:.6f}"
    )
#------------------------- SAVING BEST MODEL -------------------------
    if average_val_loss_LSTM < best_val_loss_LSTM:
        best_val_loss_LSTM = average_val_loss_LSTM
        torch.save(model_LSTM.state_dict(), "BestModel_LSTM.pth")
        print("Saved The Best Performing Model")


# wandb.finish()

Epoch [1/10] train loss: 0.690, val loss: 0.682, lr: 0.001000
Saved The Best Performing Model
Epoch [2/10] train loss: 0.662, val loss: 0.651, lr: 0.001000
Saved The Best Performing Model
Epoch [3/10] train loss: 0.583, val loss: 0.577, lr: 0.001000
Saved The Best Performing Model
Epoch [4/10] train loss: 0.426, val loss: 0.475, lr: 0.001000
Saved The Best Performing Model
Epoch [5/10] train loss: 0.253, val loss: 0.399, lr: 0.001000
Saved The Best Performing Model
Epoch [6/10] train loss: 0.139, val loss: 0.359, lr: 0.001000
Saved The Best Performing Model
Epoch [7/10] train loss: 0.075, val loss: 0.343, lr: 0.001000
Saved The Best Performing Model
Epoch [8/10] train loss: 0.045, val loss: 0.338, lr: 0.001000
Saved The Best Performing Model
Epoch [9/10] train loss: 0.030, val loss: 0.342, lr: 0.001000
Epoch [10/10] train loss: 0.022, val loss: 0.340, lr: 0.001000


In [ ]:
#Test the model
model_LSTM.load_state_dict(torch.load("BestModel_LSTM.pth"))
model_LSTM.eval()

test_loss_LSTM = 0.0
correct = 0
total = 0

with torch.no_grad():
    for batch_x, batch_y in val_loader_LSTM:
        output = model_LSTM(batch_x)
        val_loss_LSTM = criterion(output, batch_y)
        test_loss_LSTM += val_loss_LSTM.item()

        HighestValue, predicted = torch.max(output, 1)
        total += batch_y.size(0)
        correct += (predicted == batch_y).sum().item()

average_test_loss_LSTM = test_loss_LSTM / len(val_loader_LSTM)
test_acc = 100*(correct/total)


print(f"Test loss: {average_test_loss_LSTM:.3f}")
print(f"Test accuracy: {test_acc:.2f}%")

wandb.finish()

Test loss: 0.338
Test accuracy: 86.00%


## Task 1.2: Transformers Implementation
For this task, you will implement your transformer in PyTorch. 

In [ ]:
df = pd.read_csv("amazon_cells_labelled.txt", delimiter='\t', header=None)
sentiment_mapping = {0: 'Negative', 1: 'Positive'}
df['Rating'] = df[1].map(sentiment_mapping)
print(df.head())

print(df['Rating'].value_counts())

In [ ]:
class ReviewDataset(torch.utils.data.Dataset):
    def __init__(self, csv_file, tokenizer, max_length):
        self.dataset = pd.read_csv(csv_file, delimiter='\t', header=None, names=['Sentence', 'Class'])
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.label_dict = {0: 'Negative', 1: 'Positive'}
    

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        review = self.dataset.iloc[idx, 0]
        sentiment = self.dataset.iloc[idx, 1]
        label = self.label_dict[sentiment]

        encoding = self.tokenizer(
            review,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            return_attention_mask=True,
            return_tensors='pt',
            truncation=True
        )

        return {
            'review_text': review,
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(sentiment, dtype=torch.long)
        }

In [ ]:
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
review_dataset = ReviewDataset("amazon_cells_labelled.txt", tokenizer, 512)
review_dataset[0]
tokenizer.decode(review_dataset[0]['input_ids'])

In [ ]:
train_size_bert = int(0.8 * len(df))
val_size_bert = len(df) - train_size_bert
train_dataset_bert, val_dataset_bert = random_split(review_dataset, [train_size_bert, val_size_bert])

train_loader_bert = DataLoader(train_dataset_bert, batch_size=16, shuffle=True)
val_loader_bert = DataLoader(val_dataset_bert, batch_size=16, shuffle=False)

print(f"Number of training samples: {len(train_dataset_bert)}")
print(f"Number of validation samples: {len(val_dataset_bert)}")

In [ ]:
class CustomDistilBertForClassification(nn.Module):
    def __init__(self, num_labels=2):
        super(CustomDistilBertForClassification, self).__init__()
        self.distilbert = DistilBertModel.from_pretrained('distilbert-base-uncased')
        self.pre_classifier = nn.Linear(self.distilbert.config.dim, 64)
        self.dropout = nn.Dropout(0.4)
        self.classifier = nn.Linear(64, num_labels)

    def forward(self, input_ids, attention_mask):
        distilbert_output = self.distilbert(input_ids=input_ids, attention_mask=attention_mask)
        hidden_state = distilbert_output[0]  # (batch_size, sequence_length, hidden_size)
        pooled_output = hidden_state[:, 0]  # Take the [CLS] token representation
        pooled_output = self.dropout(pooled_output)
        pooled_output = self.pre_classifier(pooled_output)
        pooled_output = nn.ReLU()(pooled_output)
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)
        return logits

In [ ]:
model = CustomDistilBertForClassification()

print(model.distilbert)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

criterion = nn.CrossEntropyLoss()
# Add weight decay (L2 regularization) to reduce overfitting
optimizer = torch.optim.Adam(model.parameters(), lr=5e-5, weight_decay=0.01)

# Learning rate scheduler to reduce LR when validation loss plateaus
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=2,
    verbose=True,
    min_lr=1e-7
)

best_val_loss_bert = float('inf')
patience = 5
patience_counter = 0

model.train()
for epoch in range(num_epochs):
    # Training phase
    total_loss = 0
    for batch in train_loader_bert:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()
        logits = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    avg_train_loss = total_loss / len(train_loader_bert)
    
    # Validation phase
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch in val_loader_bert:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            logits = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(logits, labels)
            val_loss += loss.item()
    
    avg_val_loss = val_loss / len(val_loader_bert)
    model.train()
    
    # Step scheduler
    scheduler.step(avg_val_loss)
    
    # Print metrics
    print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}")
    
    # Save best model and implement early stopping
    if avg_val_loss < best_val_loss_bert:
        best_val_loss_bert = avg_val_loss
        torch.save(model.state_dict(), "BestModel_DistilBert.pth")
        print(f"Saved best model with validation loss: {avg_val_loss:.4f}")
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping triggered after {epoch+1} epochs")
            break

In [ ]:
# Test the DistilBERT model
model.load_state_dict(torch.load("BestModel_DistilBert.pth"))
model.eval()

test_loss = 0.0
correct = 0
total = 0

with torch.no_grad():
    for batch in val_loader_bert:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        logits = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = criterion(logits, labels)
        test_loss += loss.item()
        
        # Calculate accuracy
        _, predicted = torch.max(logits, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

avg_test_loss = test_loss / len(val_loader_bert)
test_accuracy = 100 * (correct / total)

print(f"Test Loss: {avg_test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.2f}%")
print(f"Correct Predictions: {correct}/{total}")

## Task 1.3 Comparison
Here, you should compare of all three models; you are requested to use the same test dataset
for Simple ANN, LSTM based model and the Transformer to answer the following:

• Compare the performance of the two models and explain in which scenarios you would
prefer one over the other.

• How did the two models’ complexity, accuracy, and efficiency differ? Did one model
outperform the other in specific scenarios or tasks? If so, why?

• What insights did you obtain concerning data amount to train? Embedding utilized?
Architectural choices made?